# 03 — Cohort-scale design from a batch of variants

**AlleleForge is a research tool. It is not a medical device and provides no medical advice.**

The same `design()` entry point that handles one variant scales to a cohort: resolve each record,
route it to every eligible chemistry, score candidates with calibrated uncertainty, run
population-aware off-target, and rank — then reduce the per-variant `RankedMenu`s to one summary
table. This notebook runs a small synthetic "VCF" end to end against the weight-free stub models, so
it is fully reproducible in CI; point it at a real VCF + hg38 reference and the loop is unchanged.

In [ ]:
import tempfile
from pathlib import Path

from alleleforge.design.designer import design
from alleleforge.genome.reference import ReferenceGenome
from alleleforge.types.edit import EditIntent
from alleleforge.types.sequence import GenomicInterval, Strand

# One engineered locus: a protospacer with an in-window A (ABE-correctable) and an NGG PAM, so the
# router can offer base editing, prime editing, and (for knock-outs) nuclease across the cohort.
PAD = "T" * 20
PROTO = "TTTAAACGTTTTTTTTTTTT"
contig = PAD + PROTO + "TGG" + PAD

fasta = Path(tempfile.mkdtemp()) / "cohort.fa"
fasta.write_text(f">chr2\n{contig}\n")
reference = ReferenceGenome(fasta, build="hg38")


def ref_base(one_based: int) -> str:
    iv = GenomicInterval(chrom="chr2", start=one_based - 1, end=one_based, strand=Strand.PLUS)
    return str(reference.fetch(iv))


print("locus length:", len(contig), "| edit position chr2:26 ref base:", ref_base(26))

## 1. A synthetic cohort

Each row stands in for a VCF record: a sample id, a 1-based position, an alternate allele, and the
editing intent. In practice these come straight from `cyvcf2`/`pysam` iteration over a `.vcf.gz`.

In [ ]:
cohort = [
    {"sample": "S1", "pos": 26, "alt": "G", "intent": EditIntent.INSTALL},
    {"sample": "S2", "pos": 26, "alt": "C", "intent": EditIntent.INSTALL},
    {"sample": "S3", "pos": 26, "alt": "G", "intent": EditIntent.KNOCK_OUT},
]
for rec in cohort:
    rec["variant"] = f"chr2:{rec['pos']}:{ref_base(rec['pos'])}>{rec['alt']}"
print("\n".join(f"{r['sample']}: {r['variant']}  ({r['intent'].value})" for r in cohort))

## 2. Design the cohort with `design_many`, then reduce to a summary

`design_many` streams a whole cohort through `design` with **bounded memory** (each `RankedMenu` is
summarized then released), a **resumable** JSONL run manifest, and **per-item failure isolation**. It
returns a `CohortRunReport` whose compact per-item summaries carry the headline facts — how many
chemistries were offered, the best candidate's chemistry, its calibrated efficiency, and the worst
off-target score across the menu. (`design_many` applies one intent per call, so we group by intent.)

In [ ]:
from alleleforge.design import design_many

# `design_many` is the cohort multiplier over `design`: it streams the input (bounded memory — each
# menu is summarized then released), is resumable via a JSONL run manifest, and isolates per-item
# failures. It applies one intent per call, so we group the cohort by intent.
summary_by_variant: dict[str, dict] = {}
for intent in sorted({r["intent"] for r in cohort}, key=lambda i: i.value):
    recs = [r for r in cohort if r["intent"] == intent]
    report = design_many(
        [r["variant"] for r in recs], reference=reference, intent=intent, run_offtarget=True
    )
    summary_by_variant.update({it.item_id: (it.summary or {}) for it in report.items})

rows = []
for rec in cohort:
    s = summary_by_variant.get(rec["variant"], {})
    rows.append(
        {
            "sample": rec["sample"],
            "intent": rec["intent"].value,
            "n_candidates": s.get("n_candidates", 0),
            "chemistries": "|".join(s.get("chemistries", [])),
            "best": s.get("best_chemistry") or "-",
            "best_eff": round(s.get("best_efficiency") or float("nan"), 2),
            "worst_offtarget": round(s.get("worst_offtarget", 0.0), 3),
        }
    )

hdr = ["sample", "intent", "n_candidates", "chemistries", "best", "best_eff", "worst_offtarget"]
print("  ".join(f"{h:>14}" if h != "chemistries" else f"{h:<28}" for h in hdr))
for r in rows:
    print("  ".join(f"{str(r[h]):<28}" if h == "chemistries" else f"{str(r[h]):>14}" for h in hdr))

## 3. Every result is reproducible

Each menu embeds a provenance block — AlleleForge version, seed, reference build, and the model
checkpoints and datasets touched — so a cohort run can be re-derived from its config and seed. This
is what makes a batch result auditable rather than a one-off.

In [ ]:
menu = design(cohort[0]["variant"], reference=reference, intent=cohort[0]["intent"])
prov = menu.provenance
print("alleleforge version :", prov.alleleforge_version)
print("seed                :", prov.seed)
print("reference build     :", prov.reference_build)
print("models invoked      :", [m.name for m in prov.models])
assert len(rows) == len(cohort)

## 4. The `cyvcf2` fast path: `iter_vcf`

`variant.iter_vcf` is the adapter that *produces* the lazy stream `design_many` consumes, straight
from a VCF: one record per **concrete ALT allele** (multi-allelic rows split), with symbolic
(`<DEL>`), spanning-`*`, and non-`PASS` calls dropped. `cyvcf2` is an optional htslib dependency, so
in production this is simply `iter_vcf("cohort.vcf.gz")`; here we feed it a tiny list of records
duck-typed to the cyvcf2 `Variant` shape so the notebook stays self-contained and runs in CI.

In [ ]:
from dataclasses import dataclass

from alleleforge.variant import iter_vcf


@dataclass
class VcfRow:
    """The slice of a ``cyvcf2.Variant`` that ``iter_vcf`` reads (1-based POS)."""

    CHROM: str
    POS: int
    REF: str
    ALT: list[str]
    ID: str | None = None
    FILTER: str | None = None  # None == PASS


vcf_rows = [
    VcfRow("chr2", 26, "A", ["G", "C"], ID="rs1"),  # multi-allelic -> two variants
    VcfRow("chr2", 26, "A", ["T"], FILTER="LowQual"),  # soft-filtered -> dropped
]
records = list(iter_vcf(vcf_rows))
print("VCF rows in:", len(vcf_rows), "-> designable records out:", len(records))
for r in records:
    print("  ", r.chrom, r.pos, f"{r.ref}>{r.alt}", "rsid=", r.rsid)

# Hand the stream straight to design_many -- the whole VCF flows through with bounded memory.
vcf_report = design_many(iter_vcf(vcf_rows), reference=reference, intent=EditIntent.INSTALL)
print("designed:", vcf_report.succeeded, "| failed:", vcf_report.failed)
assert vcf_report.succeeded == 2 and vcf_report.failed == 0

For a real cohort, wrap the VCF with `iter_vcf("cohort.vcf.gz")` and pass it straight to `design_many(..., ...)` with a
`manifest_path` (so an interrupted run resumes), an `output_dir` (durable per-sample menu JSON), and
`on_result` (stream results for `O(1)` memory); add a gnomAD database and per-sample phased
haplotypes via `gnomad=...`/`haplotypes=...` for population- and haplotype-aware off-target, and a
`reference_factory` + `max_workers` to parallelize. The orchestration above does not change — only
the data sources do.